# Week 7 - Linear classification

Learning contents:
* Part 1: Least squares for classification (Lecture 13)
* Part 2: Logistic regression (Lecture 14)
* Part 3: Multi-class logistic regression (Optional but highly recommended, Lecture 14)
* Part 4: Multi-class logistic regression on the original representation (Optional, Lecture 14)
* Part 5: Perceptron (Optional, Lecture 13)

In [ ]:
# Dependencies
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA 
import matplotlib.pyplot as plt 
import seaborn as sns; sns.set_theme() # this import just makes the plots prettier
import numpy as np

## Data
We'll be working with a dataset of handwritten digits. 
Let visualise some examples.

In [ ]:
# We're using a subset of two classes for now
digits = load_digits(n_class=2)

In [ ]:
# Handy plotting functions
x_min, x_max = -40, 40
y_min, y_max = -40, 40

def plot_examples():
    show_num = 4
    _, axes = plt.subplots(1, show_num)
    images_and_labels = list(zip(digits.images, digits.target))
    for ax, (image, label) in zip(axes[:], images_and_labels[:show_num]):
        ax.set_axis_off()
        ax.imshow(image, cmap=plt.cm.gray_r, interpolation='nearest')
        ax.set_title('Label: %i' % label)

def plot_scatter(data, target, n_class=2, alpha=0.5):
    scatter = plt.scatter(data[:, 0], data[:, 1], c=target, edgecolor='none', alpha=alpha, cmap=plt.get_cmap('rainbow', n_class))
    plt.legend(*scatter.legend_elements(), loc="upper left", title="Targets")
    plt.xlabel('Component 1')
    plt.ylabel('Component 2')
    plt.xlim(x_min, x_max)
    plt.ylim(y_min, y_max)

def plot_decision_boundary(weights, c=0.5):
    """Plot the decision boundary of a linear classifier as a line.

    Parameters
    ----------
    weights : array of shape (3,) or (1, 3) -- [w0, w1, w2], bias first
    c : scalar threshold on the model output
        (0.5 for least squares with {0,1} targets; 0 for logistic /
        perceptron activations)

    Hint (geometry): the boundary is the set of points where
        w0 + w1*x1 + w2*x2 = c.
    Solving for x2 gives a line  x2 = slope * x1 + intercept  with
        slope = -w1 / w2
        intercept = (c - w0) / w2
    Evaluate it on xx = np.linspace(x_min, x_max) and plot with
    plt.plot(xx, yy, 'k--').
    """
    weights = np.asarray(weights).flatten()
    # TODO: compute slope and intercept as in the docstring, then plot the line
    raise NotImplementedError("TODO: plot the line x2 = slope * x1 + intercept")

def plot_mesh(X, pred_fn, n_class=2):
    """Shade the decision regions of a classifier (used in the optional Part 3)."""
    h = 0.1  # step size in the mesh
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = pred_fn(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    cs = plt.contourf(xx, yy, Z, alpha = 0.1, cmap=plt.get_cmap('rainbow', n_class))
    plt.axis('tight')
    plt.xlim(x_min, x_max)
    plt.ylim(y_min, y_max)

In [ ]:
plot_examples()

In [ ]:
# The dataset contains 2D data in the form of the `images` attribute, 
# as well as a 1D-version called `data`, where the images have been flattened. 
np.array_equal(digits.images[0].flatten(), digits.data[0])

In [ ]:
# We can get a 2D version of the data using PCA
pca = PCA(n_components=2)
X = pca.fit_transform(digits.data) # this is the representation, we'll be working with

In [ ]:
# Out targets are in the set {0,1}
t_01 = digits.target

In [ ]:
# Let's plot all the data in 2D
plot_scatter(X, t_01)

## Conventions used this week

Following the lecture notation (Lectures 13-14):

* **Bias augmentation:** we prepend a constant feature $\phi_0(x) = 1$, so $\Phi = [1, X]$ has shape (N, M+1) and the weight vector $\tilde{W} = (w_0, w_1, \dots, w_M)$ includes the bias $w_0$. With the 2-D PCA data, $X$ is (360, 2) and $\Phi$ is (360, 3).
* **Target coding:** Parts 1-2 use $t \in \{0, 1\}$; Parts 3-4 use a 1-of-K (one-hot) target matrix $T$ of shape (N, K); Part 5 (perceptron) uses $t \in \{-1, +1\}$.
* $\sigma(a) = 1/(1+e^{-a})$ is the logistic sigmoid; $\eta$ is the learning rate.

Each part below states which lecture it relies on. If your lab session falls before the week's second lecture (Lecture 14), start with Parts 1 and 5 (Lecture 13 only).

## 1) Least Squares

*Uses Lecture 13, slides 13 and 15. See the Conventions cell above for notation.*

### 1.1) Find the weight vector using Least Squares for classification
Find the optimal weights **W** of size (M+1,1), where M is the size of each data point **X_n**={x_1n,...,x_Mn} , in this problem each **X_n** has size 2 because it has 2 coordinates. And where N is the number of data points. To find the optimal weight calculation look at slides 13 and 15 of Lecture 13. 
Hint: The PHI matrix will have a column of 1's so that the initial weight, **w_0**, can  be multiplied by 1. Construct PHI=[1,**X**], where **X** is the matrix of data points, in this case **X** will have dimensions [N x M] (N: total data points and M: no. of basis vectors, in this case with M=2 components/coordinates) and PHI will have dimensions [N x M+1].

In [ ]:
# TODO: Task 1.1 -- least-squares weights
def augment(X):
    """Prepend the bias feature phi_0(x) = 1 to each row.

    Input:  X of shape (N, M)          (here: (360, 2) PCA components)
    Output: X_tilde of shape (N, M+1)  (here: (360, 3)), first column all ones
    Hint: np.hstack and np.ones.
    """
    # TODO: implement
    raise NotImplementedError

X_tilde = augment(X)                   # shape (360, 3)

# TODO: compute the least-squares weights (Lecture 13, slides 13 & 15):
#   W = (Phi^T Phi)^{-1} Phi^T t  -- np.linalg.pinv(X_tilde) computes (Phi^T Phi)^{-1} Phi^T.
# Store them as a (1, 3) row vector (np.expand_dims(..., axis=0)).
W_least_squares = ...

# Self-checks:
assert X_tilde.shape == (360, 3), "X_tilde should be (360, 3)"
assert W_least_squares.shape == (1, 3), "W_least_squares should be (1, 3)"


### 1.2) Create class predictions using the weight vector
Create a function called 'predict' that takes as arguments: the weights vector, **W**, found using the Least Squares method for classification in the previous step, the augmented data created by adding a row of 1's to the data, **X**, and a decision boundary, which will be scalar (choose the value 0.5) that will be used to determine the boundary of classification. The function should return the predictions for each data point, so it should return an array of [Nx1] values, based on the decision boundary value for classification. See slide 13, Lecture 13. Finally, check that the predictions are the same than the original target vector.

In [ ]:
# TODO: Task 1.2 -- class predictions from the linear model
def predict(W, X_tilde, boundary=0.5):
    """Predict class labels {0, 1} from a linear model.

    Inputs:
      W        : (1, M+1) weight row vector
      X_tilde  : (N, M+1) bias-augmented data
      boundary : scalar threshold on the model output y = W @ X_tilde.T
                 (0.5 for least squares with {0,1} targets)
    Output:
      preds    : (N,) integer array with values in {0, 1}
    """
    # TODO: compute y = W @ X_tilde.T and threshold it at `boundary`
    raise NotImplementedError

preds = predict(W_least_squares, X_tilde)
print(np.array_equal(t_01, preds))     # should print True (all 360 correct)


### 1.3) Plot the decision boundary 
You will need to write a function named plot_decision_boundary() earlier, to plot the decision boundary of the linear classifier. Once you have defined the function, here you will call it by providing **W_least_squares** as parameter obtained in the previous step, to show the plot.

In [ ]:
plot_scatter(X, t_01)
# TODO: uncomment once plot_decision_boundary (top of the notebook) is implemented:
# plot_decision_boundary(W_least_squares, c=0.5)

## 2) Logistic Regression

*Uses Lecture 14, slide 20 — if your lab falls before Lecture 14, do Parts 1 and 5 first.*
### 2.1) Find the weight vector using the Logistic Regression
To find the weight vector, compute the gradient of the error function with respect to **w**, ∇E(**w**) (see slide 20 of Lecture 14). **PHI(X)** has dimensions [Nx(M+1)], so it only has an added row of 1's, in this case it has dimension 360x3. Then, starting from the provided randomly-initialized weight vector `W_0` (see the next cell; not to be confused with the bias component $w_0$), perform the weight update for 30 epochs: $w^{(\tau+1)} = w^{(\tau)} - \eta\,\nabla E(w^{(\tau)})$.

Prepare the PHI matrix the same way as you did in section 1.1

Use the formulas of the logistic regression method given in Section 4.3.2 in the text book

In [ ]:
# initialise W_0 randomly
np.random.seed(42)
W_0 = 2 * np.random.random((1, 3)) -1 # random values in the range [-1,1]

# TODO: build the PHI matrix as in section 1.1 (PHI = augment(X), shape (360, 3))

# corresponding target vectors should be {0, 1}
t_01 = digits.target

In [ ]:
learning_rate = 0.1

# TODO: Task 2.1 -- gradient-descent training (Lecture 14, slide 20)
def sigmoid(a):
    """Logistic sigmoid sigma(a) = 1 / (1 + exp(-a)); works elementwise on arrays."""
    # TODO: implement
    raise NotImplementedError

def error(W, PHI, t):
    """Gradient of the cross-entropy error for logistic regression.

    Inputs:  W (1, M+1), PHI (N, M+1), t (N,) with values in {0, 1}
    Output:  gradient of shape (M+1,) (or (1, M+1)):
             (1/N) * PHI^T (y - t),  where  y = sigma(W @ PHI.T)
             are the predicted probabilities.
    """
    # TODO: implement
    raise NotImplementedError

# TODO: starting from W_0, do 30 gradient-descent updates:
#   W <- W - learning_rate * error(W, PHI, t_01)
W_logistic = ...


### 2.2) Perform class-predictions
Use weights obtained in the previous step along with the input matrix X to obtain the model predictions on the input data X. Remember that the prediction formula will be the same as used in the Least Squares based model (slide 13 of Lecture 13).

Note: for logistic regression the exact decision rule is $\sigma(a) \geq 0.5$, which is equivalent to thresholding the linear activation $a = W\phi(x)$ at 0 — so call `predict` with `boundary=0`.

In [ ]:
# TODO: Task 2.2 -- predictions with the logistic weights
# Reuse predict() from Task 1.2. For logistic regression, sigma(a) >= 0.5
# exactly when the linear activation a >= 0, so use boundary=0.
preds_logistic = ...
print(np.array_equal(t_01, preds_logistic))   # should print True (all 360 correct)


### 2.3) Plot the decision boundary
The function written earlier for Least squares classifier, plot_decision_boundary(), will again be used to plot the decision boundary

In [ ]:
plot_scatter(X, t_01)
# TODO: uncomment once plot_decision_boundary (top of the notebook) is implemented:
# plot_decision_boundary(W_logistic, c=0)

## 3) Multi-class logistic regression (Optional but Highly Recommended)

*Uses Lecture 14 (multi-class logistic regression; textbook section 4.3.4).*
Repeat 2) but now for multiple classes, i.e. compute the weight matrix, perform predictions (you should be able to get about 93% accuracy) and plot decision boundaries.

Hint: You will need to use a one-hot encoding of the targets. See section 4.3.4 in the textbook.

In [ ]:
# Data
n_class = 3
digits = load_digits(n_class=n_class)
pca = PCA(n_components=2)
X_mult = pca.fit_transform(digits.data)

learning_rate = 0.1

# We need to do a one_hot encoding of our data:
# I.e. 0 -> [1,0,0], 1 -> [0,1,0], 2 -> [0,0,1]
def one_hot(targets, n_class=n_class):
    res = np.eye(n_class)[np.array(targets).reshape(-1)]
    return res.reshape(list(targets.shape)+[n_class])

t_oh = one_hot(digits.target, n_class)

In [ ]:
plot_scatter(X_mult, digits.target, n_class=n_class)

In [ ]:
# TODO (optional): Task 3 -- multi-class logistic regression (Lecture 14; PRML section 4.3.4)

def softmax(a):
    """Softmax over the class dimension (axis 0 for a (K, N) activation array).
    Hint: subtract np.max(a) before exponentiating to prevent overflow."""
    # TODO: implement
    raise NotImplementedError

def grad_error(W, PHI, T):
    """Gradient of the multi-class cross-entropy: (1/N) * PHI^T (Y - T),
    reshaped to W's shape.  W: (K, M+1), PHI: (N, M+1), T: one-hot (N, K)."""
    # TODO: implement
    raise NotImplementedError

def predict_multi(W, PHI):
    """Class predictions: argmax over the softmax outputs. Returns (N,) ints."""
    # TODO: implement
    raise NotImplementedError

def accuracy(preds, targets):
    return np.sum(preds == targets) / len(targets)

# TODO: build PHI = augment(X_mult); initialise the (n_class, 3) weight matrix with
#   np.random.seed(25); W_0 = 2 * np.random.random((n_class, 3)) - 1
# then do 50 gradient-descent updates  W <- W - learning_rate * grad_error(W, PHI, t_oh).
# Expected: accuracy(predict_multi(W, PHI), digits.target) is about 0.93.
# Decision regions: plot_scatter(X_mult, digits.target, n_class=n_class) followed by
#   plot_mesh(X_mult, lambda x: predict_multi(W, augment(x)), n_class=n_class)


## 4) Multi-class logistic regression on original data representation (Optional Exercise)

*Uses Lecture 14.*
Repeat exercise 3), but use the original data representation instead.
You can omit the plots.

Hint: To get 100% accuracy, you can try to:
- Use the feature map $\phi(X) = [1, X, X^2]$ — bias, the 64 original pixel values, **and their squares** (129 features in total). With only $\phi(X) = [1, X]$ you will plateau below 100%.
- Rescale your data to the range [0,1]
- Run 1000 epochs with learning_rate = 0.1

In [ ]:
# TODO (optional): Task 4 -- repeat Task 3 on the original 64-pixel representation.
# Follow the hints above: phi(X) = [1, X, X**2] on rescaled data (digits.data / digits.data.max()),
# seed-25 initialisation of shape (n_class, 129), 1000 epochs, learning_rate = 0.1.
# Expected: accuracy = 1.0. (Plots can be omitted.)


## 5) Perceptron (Optional Exercise)

*Uses Lecture 13, slide 20.*

### 5.1) set up and implement the Perceptron forward model 
Implement the perceptron algorithm for classification (see slide 20 of Lecture 13) for 1000 epochs. For this, consider PHI(X) as a polynomial basis of order D=1, this means PHI(X)_n = { 1, X }, so PHI(X) will have dimensions [Nx(MxD+1)]. In this case D=1 (polynomial of order 1), M=2 (each data point X_n has two coordinates) and N is 360 data points. Assume that the data is already suffled. To initialize the weight vector w_0, we set the random seed to 13 and make that the weight vector is in the range of {-1,1}. In addition, a learning_rate of 0.1 should do the job.
Hint: remember to change the target representation range so that each data point will have a class of -1 or 1.

In [ ]:
# TODO: Task 5.1 -- the perceptron algorithm (Lecture 13, slide 20)
# Data: the 2-class digits from Part 1 (N=360). Rebuild it here so this part
# runs independently of Parts 3-4 (which overwrote `digits`).
digits2 = load_digits(n_class=2)
X_2 = PCA(n_components=2).fit_transform(digits2.data)   # (360, 2)
t_01_p5 = digits2.target                                # targets in {0, 1}

# Perceptron target coding: the classes must be -1 and +1.
# TODO: map t_01_p5 from {0, 1} to {-1, +1}
t_pm = ...

# TODO: bias-augmented data [1, X_2], shape (360, 3) -- reuse augment() from Task 1.1
PHI_p5 = ...

# Weight initialisation: random values in [-1, 1], seed 13
np.random.seed(13)
w_perceptron = 2 * np.random.random(3) - 1

learning_rate = 0.1

# TODO: for up to 1000 epochs, loop over the data points; if point n is
# misclassified (f(w @ phi_n) != t_n with f(a) = +1 if a >= 0 else -1), update
#   w <- w + learning_rate * phi_n * t_n        (Lecture 13, slide 20)
# The data is linearly separable, so the perceptron converges (here within a
# few epochs); you can break out of the loop once a full epoch makes no updates.


### 5.2) Perform class-predictions
Again, you should be able to classify all correctly by checking that the predictions are the same as the provided target vector. Notice both predictions and target vector are an array of N components of 0's and 1's, so you have to make your predictions go back to 1's and 0's instead of -1's and 1's.

In [ ]:
# TODO: Task 5.2 -- perceptron predictions
# Predict with f(a) = +1 if a >= 0 else -1, where a = PHI_p5 @ w_perceptron,
# then map the predictions from {-1, +1} back to {0, 1} and compare with t_01_p5.
preds_p5 = ...
print(np.array_equal(t_01_p5, preds_p5))      # should print True (all 360 correct)


### 5.3) Plot the decision boundary
To plot the decision boundary, adapt the names from your weights and PHI matrix in the code below.

In [ ]:
# Task 5.3 -- plot the decision boundary
# The perceptron thresholds the activation at 0, so pass c=0.
plot_scatter(X_2, t_01_p5)
# TODO: uncomment once plot_decision_boundary (top of the notebook) is implemented:
# plot_decision_boundary(w_perceptron, c=0)
